# 25. API로 LLM 사용하기

> **제25장** · **이론편 대응: 20.6절(대화 형식), 22.1절(오픈 가중치 vs API)**
> **예상 소요**: 70분
> **필요 사양**: **[CPU]** — 인터넷 연결 필요
> **추가 설치**: **openai, python-dotenv** (1절 참조)
> **API 키**: 필요 (1절에서 무료 발급 방법 안내)

---

## 이 장에서 하는 일

23~24장에서는 모델을 내려받아 직접 돌렸다. 이번에는 **남의 서버에 있는 모델**을 쓴다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **준비 — API 키 발급과 보관** | 22.1절 |
| 2 | 첫 호출 | — |
| 3 | **ChatML이 API에서는 어떻게** | 20.6절 |
| 4 | 생성 파라미터 (24장과 대조) | 20.5절 |
| 5 | 스트리밍 | — |
| 6 | 토큰 사용량과 비용 | 20.2절 |
| 7 | 구조화된 출력 | 20.6절 |
| 8 | 오류 처리와 재시도 | — |

**API 키가 없어도 1~3절의 개념 설명과 코드 구조는 확인할 수 있게** 구성했다.
실제 호출이 필요한 셀은 키가 없으면 안내를 출력하고 넘어간다.

---

## 1. 준비 — API 키 발급과 보관

### 1-1. 왜 API를 쓰나 — 이론편 22.1절

이론편 22.1절에서 오픈 가중치 모델과 API 모델을 비교했다. 실무 관점에서 정리하면 이렇다.

| 항목 | 로컬 모델 (23~24장) | API |
|---|---|---|
| 준비 | 모델 내려받기, GPU 필요 | 키만 있으면 즉시 |
| 성능 | 내 장비 수준까지 | 최상급 모델 사용 가능 |
| 비용 | 장비 구입·전기 | 호출량만큼 과금 |
| 데이터 | 외부로 안 나감 | 제공자에게 전송됨 |
| 속도 | 장비에 좌우 | 네트워크 지연 있음 |
| 파인튜닝 | 자유롭게 (31~32장) | 제한적 |

**둘 중 하나를 고르는 문제가 아니다.** 실무에서는 섞어 쓰는 경우가 많다 —
민감한 데이터는 로컬로, 어려운 작업은 API로.

> **구독과 API 키는 다르다.** 헷갈리기 쉬운 부분이다.
>
> | 구분 | 용도 | 결제 |
> |---|---|---|
> | 구독 | 웹·앱에서 대화, 코딩 도구 사용 | 월정액 |
> | **API 키** | **코드에서 호출 (이 장의 실습)** | 사용량만큼 |
>
> 챗봇 서비스를 구독 중이더라도 API 키는 **따로 발급**받아야 한다.

### 1-2. 무료로 시작하기 — 카드 없이

**OpenAI는 2026년 기준 신용카드 등록이 필수**이며 무료 크레딧이 없다.
학습 목적이라면 카드 없이 쓸 수 있는 곳부터 시작하는 편이 낫다.

| 제공처 | 카드 | 특징 |
|---|---|---|
| **Google AI Studio** | 불필요 | 긴 문맥, 멀티모달 |
| **Groq** | 불필요 | 매우 빠른 응답 |
| **OpenRouter** | 불필요 | 여러 모델을 하나의 키로 |
| **GitHub Models** | 불필요 | GitHub 계정만 있으면 |
| OpenAI | **필요** | — |

> **중요**: 무료 한도와 조건은 자주 바뀐다. 실제로 쓰기 전에 각 제공처의 공식 페이지에서
> 현재 조건을 확인해야 한다. 무료 티어는 학습·검증용이며 운영 서비스에는 적합하지 않다.

**대부분이 OpenAI 호환 엔드포인트를 제공한다**는 점이 핵심이다.
`base_url`만 바꾸면 같은 코드가 그대로 돌아간다. 2절에서 확인한다.

### 1-3. OpenAI 발급 절차

카드를 등록해 OpenAI를 쓰려면 다음 순서를 따른다.

**1단계 — 계정 만들기**

`platform.openai.com` 에 접속해 가입한다. 구글·마이크로소프트 계정 연동도 가능하다.

> ChatGPT 구독(Plus)과 **API는 별개**다. Plus를 쓰고 있어도 API는 따로 결제해야 한다.

**2단계 — 결제 수단 등록**

`Settings → Billing → Add payment details`

개인/사업자를 고르고 카드 정보를 입력한다. 이 단계를 건너뛰면 키를 만들어도
호출 시 다음 오류가 난다.

```
BadRequestError: 400 Billing hard limit has been reached
```

**3단계 — 크레딧 충전**

선불 방식이다. 최소 $5부터 충전할 수 있다. 충전한 만큼만 쓰이므로
예상치 못한 요금이 나올 걱정이 적다.

**4단계 — 키 생성**

`API Keys → Create new secret key`

이름을 붙이고 만든다. 프로젝트별로 다른 키를 만들어 두면 나중에 사용량을 구분해 볼 수 있다.

> **키는 생성 직후 한 번만 표시된다.** 창을 닫으면 다시 볼 수 없으므로 즉시 복사해 둔다.
> 잃어버렸다면 새로 만들고 기존 것은 삭제하면 된다.

**5단계 — 사용량 한도 설정 (권장)**

`Settings → Limits` 에서 월 한도를 정할 수 있다. 코드 실수로 반복 호출이 일어나도
정해둔 금액을 넘지 않는다.

### 1-4. 키 보관 — 절대 코드에 쓰지 않는다

**가장 흔하고 위험한 실수**가 키를 코드에 직접 쓰는 것이다.

```python
# 절대 이렇게 하지 말 것
client = OpenAI(api_key="sk-proj-abc123...")
```

저장소에 올라가면 **자동 수집 프로그램이 곧바로 찾아낸다.** 실제로 이렇게 유출된 키로
수백 달러가 청구되는 사례가 계속 보고된다.

**올바른 방법: `.env` 파일에 두고 읽어 쓴다.**

1. 프로젝트 루트에 `.env` 파일을 만든다

```
OPENAI_API_KEY=sk-proj-여기에실제키
```

2. `.gitignore`에 `.env`를 넣는다 (이 저장소는 이미 되어 있다)

3. 코드에서는 이렇게 읽는다

```python
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
```

**`.env.example` 파일도 함께 두는 것이 관행이다.** 값은 비워 두고 어떤 키가 필요한지만 알려주는 용도다.
이 저장소에도 있으니 복사해서 쓰면 된다.

```
cp .env.example .env
```

In [ ]:
import importlib
from pathlib import Path

print("=" * 60)
print("필요 패키지 확인")
print("=" * 60)

required = [
    ("openai", "OpenAI 호환 클라이언트", "pip install openai"),
    ("dotenv", "환경 변수 읽기", "pip install python-dotenv"),
]

missing = []
for name, desc, install in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "설치됨")
        print(f"[OK]   {name:<12}{ver:<14}{desc}")
    except ImportError:
        print(f"[없음] {name:<12}{'':<14}{desc}")
        missing.append(install)

print("-" * 60)
if missing:
    print("설치가 필요합니다:")
    for cmd in missing:
        print(f"  {cmd}")
    print()
    print("설치 후 커널을 재시작하세요.")
else:
    print("[준비 완료]")

# .env 파일 확인
root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent

env_file = root / ".env"
env_example = root / ".env.example"

print()
print("=" * 60)
print("설정 파일 확인")
print("=" * 60)
print(f"프로젝트 루트: {root}")
print(f"  .env         : {'있음' if env_file.exists() else '없음'}")
print(f"  .env.example : {'있음' if env_example.exists() else '없음'}")

if not env_file.exists():
    print()
    print("  .env 를 만들려면:")
    print(f"    1) {env_example} 를 복사해 .env 로 이름 변경")
    print("    2) 발급받은 키를 입력")

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent

try:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
    HAS_DOTENV = True
except ImportError:
    HAS_DOTENV = False

print("=" * 60)
print("API 키 확인")
print("=" * 60)

# 여러 제공처를 지원 — 있는 것을 쓴다
providers = [
    ("OPENAI_API_KEY",   "OpenAI",          None),
    ("GROQ_API_KEY",     "Groq",            "https://api.groq.com/openai/v1"),
    ("OPENROUTER_API_KEY","OpenRouter",     "https://openrouter.ai/api/v1"),
    ("GEMINI_API_KEY",   "Google AI Studio", "https://generativelanguage.googleapis.com/v1beta/openai/"),
]

API_KEY = None
BASE_URL = None
PROVIDER = None

for env_name, label, base in providers:
    key = os.getenv(env_name)
    if key:
        masked = key[:8] + "..." + key[-4:] if len(key) > 12 else "***"
        print(f"[발견] {label:<18}{env_name:<22}{masked}")
        if API_KEY is None:
            API_KEY, BASE_URL, PROVIDER = key, base, label
    else:
        print(f"[없음] {label:<18}{env_name}")

print("-" * 60)
if API_KEY:
    print(f"사용할 제공처: {PROVIDER}")
    if BASE_URL:
        print(f"  base_url: {BASE_URL}")
else:
    print("API 키가 설정되지 않았습니다.")
    print()
    print(".env 파일에 아래 중 하나를 넣으세요:")
    for env_name, label, _ in providers:
        print(f"  {env_name}=발급받은키    # {label}")
    print()
    print("키가 없어도 이 장의 개념 설명과 코드 구조는 확인할 수 있습니다.")
    print("실제 호출이 필요한 셀은 안내를 출력하고 넘어갑니다.")

### 1-5. 비용 관리

API는 **토큰 단위로 과금**된다. 이론편 20.2절에서 다룬 토큰이 그대로 비용 단위가 되는 것이다.

| 구분 | 뜻 |
|---|---|
| 입력 토큰 | 내가 보낸 프롬프트 |
| 출력 토큰 | 모델이 생성한 응답 |

보통 **출력이 입력보다 비싸다.** 생성에 계산이 더 들기 때문이다.

**비용을 줄이는 방법**

| 방법 | 설명 |
|---|---|
| `max_tokens` 설정 | 출력 길이를 제한 |
| 프롬프트 줄이기 | 불필요한 설명 제거 |
| 작은 모델 사용 | 쉬운 작업은 저렴한 모델로 |
| 대화 기록 정리 | 매번 전체를 보내면 누적된다 |

**마지막 항목이 특히 중요하다.** 대화형 API는 **상태를 기억하지 않으므로**,
매 호출마다 이전 대화를 전부 다시 보내야 한다. 대화가 길어질수록 비용이 급격히 는다.
6절에서 실제로 계산해 본다.

---

## 2. 첫 호출

준비가 끝났으면 실제로 불러 본다.

**OpenAI 호환 API의 기본 형태**는 이렇다.

```python
from openai import OpenAI

client = OpenAI(api_key=..., base_url=...)   # base_url은 OpenAI면 생략

response = client.chat.completions.create(
    model="모델이름",
    messages=[{"role": "user", "content": "질문"}],
)
print(response.choices[0].message.content)
```

**`base_url` 하나로 제공처를 바꿀 수 있다는 점**이 핵심이다.

In [ ]:
print("=" * 65)
print("제공처별 설정 — base_url 만 다르다")
print("=" * 65)

configs = {
    "OpenAI":           (None, "gpt-4o-mini"),
    "Groq":             ("https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
    "OpenRouter":       ("https://openrouter.ai/api/v1", "<모델명>"),
    "Google AI Studio": ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash"),
}

print(f"{'제공처':<20}{'base_url':<52}")
print("-" * 65)
for name, (url, model) in configs.items():
    shown = url if url else "(기본값 — 생략 가능)"
    print(f"{name:<20}{shown}")
print("-" * 65)
print()
print("모델 이름은 제공처마다 다르고 자주 바뀐다.")
print("각 제공처의 문서에서 현재 사용 가능한 모델을 확인해야 한다.")
print()
print("코드 형태")
print("""
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="...",          # 제공처에 맞게
)
""")

In [ ]:
import os

def get_client():
    """설정된 키로 클라이언트를 만든다. 키가 없으면 None."""
    if not API_KEY:
        return None
    try:
        from openai import OpenAI
    except ImportError:
        print("[안내] openai 패키지가 없습니다: pip install openai")
        return None

    kwargs = {"api_key": API_KEY}
    if BASE_URL:
        kwargs["base_url"] = BASE_URL
    return OpenAI(**kwargs)


def ask(messages, model=None, **kwargs):
    """간단한 호출 래퍼. 키가 없으면 안내만 출력."""
    client = get_client()
    if client is None:
        print("[건너뜀] API 키가 없어 실제 호출을 하지 않습니다.")
        print("  1절을 참고해 .env 에 키를 설정하면 이 셀이 동작합니다.")
        return None

    model = model or DEFAULT_MODEL
    try:
        # ── chat.completions.create() 파라미터 ───────────────────────
        #   model        모델 이름.  **필수**
        #   messages     대화 기록.  **필수**
        #                [{"role": "system"|"user"|"assistant", "content": ...}]
        #   max_tokens   생성할 최대 토큰.  기본값 모델별 상이
        #                비용과 직결 — 넉넉히 잡되 상한을 둔다
        #   temperature  0.0~2.0.  기본값 1.0
        #                0.0: 결정적(사실 질의) / 0.7: 균형 / 1.2: 창의적
        #   top_p        누적 확률.  기본값 1.0
        #                temperature 와 **둘 중 하나만** 조절하는 것이 관행
        #   n            생성할 응답 수.  기본값 1
        #                Self-Consistency 에 쓴다 (비용도 n배)
        #   stream       스트리밍 여부.  기본값 False
        #                True 면 토큰 단위로 받아 체감 속도 개선
        #   stop         중단 문자열.  기본값 None.  예: ["\n\n", "###"]
        #   presence_penalty   새 주제 유도.  기본값 0.  범위 -2.0~2.0
        #   frequency_penalty  반복 억제.  기본값 0.  범위 -2.0~2.0
        #   response_format    출력 형식.  {"type": "json_object"} 가능
        #   tools        도구 목록 (함수 호출)
        #   seed         재현성용 시드.  기본값 None
        # ──────────────────────────────────────────────────────────────
        return client.chat.completions.create(
            model=model, messages=messages, **kwargs)
    except Exception as e:
        print(f"[오류] {type(e).__name__}: {str(e)[:200]}")
        return None


# 제공처별 기본 모델 (실제 사용 가능한 이름은 문서에서 확인)
DEFAULT_MODELS = {
    "OpenAI": "gpt-4o-mini",
    "Groq": "llama-3.3-70b-versatile",
    "Google AI Studio": "gemini-2.0-flash",
    "OpenRouter": "meta-llama/llama-3.3-70b-instruct",
}
DEFAULT_MODEL = DEFAULT_MODELS.get(PROVIDER, "gpt-4o-mini")

print("=" * 60)
print("첫 호출")
print("=" * 60)
print(f"제공처: {PROVIDER or '(없음)'}")
print(f"모델  : {DEFAULT_MODEL}")
print()

resp = ask([{"role": "user", "content": "안녕하세요. 한 문장으로 자기소개 해주세요."}])

if resp:
    print("응답")
    print(f"  {resp.choices[0].message.content}")
    print()
    print("메타 정보")
    print(f"  모델      : {resp.model}")
    print(f"  종료 이유  : {resp.choices[0].finish_reason}")
    if resp.usage:
        print(f"  입력 토큰  : {resp.usage.prompt_tokens}")
        print(f"  출력 토큰  : {resp.usage.completion_tokens}")

### 응답 구조

응답 객체에서 무엇을 꺼낼 수 있는지 정리한다.

```python
response.choices[0].message.content   # 실제 답변
response.choices[0].finish_reason     # 왜 끝났는가
response.usage.prompt_tokens          # 입력 토큰 수
response.usage.completion_tokens      # 출력 토큰 수
response.model                        # 실제 사용된 모델
```

**`finish_reason`을 확인하는 습관**을 들이면 좋다.

| 값 | 뜻 |
|---|---|
| `stop` | 정상적으로 답을 마침 |
| `length` | **`max_tokens`에 걸려 잘림** |
| `content_filter` | 안전 정책에 걸림 |

`length`가 나왔다면 답이 중간에 끊긴 것이다. 그대로 쓰면 안 된다.

---

## 3. ChatML이 API에서는 어떻게 — 이론편 20.6절

24장 5절에서 ChatML 형식을 직접 만들어 봤다.

```
<|im_start|>system
당신은 친절한 AI 도우미입니다.<|im_end|>
<|im_start|>user
파이썬이 뭐야?<|im_end|>
<|im_start|>assistant
```

**API에서는 이것을 리스트로 표현한다.** 특수 토큰을 직접 다룰 필요가 없다.

In [ ]:
import json

messages = [
    {"role": "system", "content": "당신은 친절한 AI 도우미입니다."},
    {"role": "user", "content": "파이썬이 뭐야?"},
    {"role": "assistant", "content": "파이썬은 배우기 쉬운 프로그래밍 언어입니다."},
    {"role": "user", "content": "어디에 쓰여?"},
]

print("=" * 65)
print("API 형식 (딕셔너리 리스트)")
print("=" * 65)
print(json.dumps(messages, ensure_ascii=False, indent=2))
print()

print("=" * 65)
print("24장에서 만든 ChatML 문자열")
print("=" * 65)
for m in messages:
    print(f"<|im_start|>{m['role']}")
    print(f"{m['content']}<|im_end|>")
print("<|im_start|>assistant")
print()
print("-" * 65)
print("같은 것을 다르게 표현했을 뿐이다.")
print()
print("API 쪽이 편한 이유")
print("  - 특수 토큰을 외울 필요가 없다")
print("  - 모델마다 다른 형식을 서버가 알아서 맞춘다")
print("  - 오타로 형식이 깨질 일이 없다")

In [ ]:
print("=" * 65)
print("중요: API는 이전 대화를 기억하지 않는다")
print("=" * 65)
print()
print("매 호출이 독립적이다. 대화를 이어가려면 **전체 기록을 매번 보내야 한다.**")
print()
print("[잘못된 방식]")
print("""
ask([{"role": "user", "content": "내 이름은 홍길동이야"}])
ask([{"role": "user", "content": "내 이름이 뭐라고?"}])   # 모른다
""")
print("[올바른 방식]")
print("""
history = []
history.append({"role": "user", "content": "내 이름은 홍길동이야"})
r = ask(history)
history.append({"role": "assistant", "content": r.choices[0].message.content})
history.append({"role": "user", "content": "내 이름이 뭐라고?"})
r = ask(history)   # 이제 안다
""")
print("-" * 65)
print("이것이 비용과 직결된다 —")
print("대화가 길어질수록 매 호출의 입력 토큰이 계속 늘어난다. (6절에서 계산)")

In [ ]:
print("=" * 60)
print("대화 이어가기 실습")
print("=" * 60)

history = [
    {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
]

turns = [
    "3 더하기 5는?",
    "거기에 2를 곱하면?",      # 앞의 답을 알아야 풀 수 있다
]

for user_msg in turns:
    history.append({"role": "user", "content": user_msg})
    print(f"\n[사용자] {user_msg}")

    resp = ask(history, max_tokens=100)
    if resp is None:
        print("  (키가 없어 이후 대화는 건너뜁니다)")
        break

    answer = resp.choices[0].message.content
    print(f"[모델]   {answer}")
    print(f"         (입력 {resp.usage.prompt_tokens} 토큰)")

    # 응답을 기록에 추가해야 다음 턴에서 참고한다
    history.append({"role": "assistant", "content": answer})

print()
print("-" * 60)
print(f"최종 대화 기록: {len(history)}개 메시지")
print("두 번째 질문에서 입력 토큰이 늘어난 것을 확인하자.")

---

## 4. 생성 파라미터 — 24장과 대조

24장에서 다룬 `temperature`, `top_p`가 API에서도 같은 이름으로 쓰인다.
**같은 개념이 그대로 적용된다.**

In [ ]:
print("=" * 68)
print("주요 파라미터 (24장과 대조)")
print("=" * 68)
print(f"{'파라미터':<18}{'20번(로컬)':<22}{'API':<16}{'설명'}")
print("-" * 68)
rows = [
    ("temperature", "temperature", "동일", "분포의 뾰족함"),
    ("top_p",       "top_p",       "동일", "누적 확률 자르기"),
    ("max_tokens",  "max_new_tokens", "이름 다름", "최대 출력 길이"),
    ("stop",        "(없음)",       "API 전용", "이 문자열이 나오면 중단"),
    ("seed",        "manual_seed", "일부 지원", "재현성"),
    ("n",           "num_return_sequences", "이름 다름", "여러 개 생성"),
]
for a, b, c, d in rows:
    print(f"{a:<18}{b:<22}{c:<16}{d}")
print("-" * 68)
print()
print("주의: top_k 는 지원하지 않는 API가 많다.")
print("      제공처마다 지원 범위가 다르므로 문서를 확인해야 한다.")

In [ ]:
print("=" * 65)
print("온도에 따른 응답 비교")
print("=" * 65)

prompt = [{"role": "user", "content": "'바다'로 시작하는 짧은 문장 하나만 써주세요."}]

for temp in [0.0, 1.0, 1.5]:
    print(f"\n[temperature = {temp}]")
    for trial in range(2):
        resp = ask(prompt, temperature=temp, max_tokens=60)
        if resp is None:
            break
        print(f"  {trial+1}회: {resp.choices[0].message.content.strip()}")
    if resp is None:
        break

print()
print("-" * 65)
print("temperature=0 이면 거의 같은 답이 나온다 (완전히 같지는 않을 수 있음).")
print("높을수록 다양해지지만 엉뚱해질 수도 있다 — 24장 2절에서 본 그대로다.")

### `stop` — API에만 있는 것

특정 문자열이 나오면 생성을 멈추게 할 수 있다.

```python
response = client.chat.completions.create(
    model=...,
    messages=...,
    stop=["\n\n", "###"],     # 이것이 나오면 중단
)
```

목록 하나만 뽑거나, 정해진 구분자까지만 받고 싶을 때 유용하다.
**출력 토큰을 줄여 비용도 아낀다.**

---

## 5. 스트리밍

지금까지는 **답이 다 완성될 때까지 기다렸다.** 긴 답변이면 수십 초가 걸린다.

스트리밍을 쓰면 **생성되는 대로 조각조각 받을 수 있다.** ChatGPT 화면에서
글자가 하나씩 나타나는 것이 이 방식이다.

23장에서 봤듯 모델은 원래 토큰을 하나씩 만든다. 스트리밍은
그것을 기다렸다 한 번에 주는 대신 **만들어지는 즉시 보내는 것**이다.

In [ ]:
import time

print("=" * 60)
print("일반 호출 vs 스트리밍")
print("=" * 60)

question = [{"role": "user", "content": "인공지능을 세 문장으로 설명해주세요."}]

client = get_client()
if client is None:
    print("[건너뜀] API 키가 필요합니다.")
    print()
    print("스트리밍 코드 형태")
    print("""
stream = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=True,              # 이 옵션 하나
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
""")
else:
    # --- 일반 호출 ---
    print("[일반 호출] 완성될 때까지 대기")
    t0 = time.time()
    resp = ask(question, max_tokens=200)
    if resp:
        wait_time = time.time() - t0
        print(f"  {wait_time:.1f}초 기다린 뒤 한 번에 받음")
        print(f"  {resp.choices[0].message.content[:80]}...")

    # --- 스트리밍 ---
    print()
    print("[스트리밍] 만들어지는 대로 받음")
    t0 = time.time()
    first_chunk_time = None
    full = []

    try:
        stream = client.chat.completions.create(
            model=DEFAULT_MODEL, messages=question,
            max_tokens=200, stream=True)

        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                if first_chunk_time is None:
                    first_chunk_time = time.time() - t0
                full.append(delta)
                print(delta, end="", flush=True)

        total = time.time() - t0
        print()
        print()
        print(f"  첫 글자까지: {first_chunk_time:.2f}초")
        print(f"  전체 완료  : {total:.2f}초")
        print(f"  조각 개수  : {len(full)}")
    except Exception as e:
        print(f"[오류] {type(e).__name__}: {str(e)[:150]}")

print()
print("-" * 60)
print("스트리밍의 이점")
print("  사용자가 기다리는 체감 시간이 크게 준다.")
print("  전체 소요 시간은 같지만, 첫 반응이 훨씬 빠르다.")
print()
print("주의: 스트리밍에서는 usage(토큰 수)가 마지막에 오거나 없을 수 있다.")

---

## 6. 토큰 사용량과 비용 — 이론편 20.2절

23장 4절에서 **한국어가 영어보다 토큰을 많이 쓴다**는 것을 봤다.
API에서는 이것이 **곧바로 비용**이 된다.

In [ ]:
print("=" * 70)
print("토큰 사용량 확인")
print("=" * 70)

samples = [
    ("영어",   "Explain artificial intelligence in one sentence."),
    ("한국어", "인공지능을 한 문장으로 설명해주세요."),
]

for lang, text in samples:
    resp = ask([{"role": "user", "content": text}], max_tokens=100)
    if resp is None:
        break
    u = resp.usage
    print(f"\n[{lang}] {text}")
    print(f"  입력 토큰: {u.prompt_tokens:4}   출력 토큰: {u.completion_tokens:4}   "
          f"합계: {u.total_tokens:4}")
    print(f"  응답: {resp.choices[0].message.content[:60]}...")

if resp is None:
    print()
    print("키가 없어 실제 측정은 건너뜁니다.")
    print("23장 4절에서 확인한 대로, 한국어가 영어보다 토큰을 많이 씁니다.")

In [ ]:
print("=" * 70)
print("대화가 길어질 때 비용 — 직접 계산")
print("=" * 70)
print()
print("가정: 매 턴마다 사용자 50토큰, 모델 200토큰")
print("      API는 상태를 기억하지 않으므로 매번 전체를 다시 보낸다")
print()

user_tokens = 50
assistant_tokens = 200

print(f"{'턴':<6}{'이번 입력':<14}{'이번 출력':<14}{'누적 입력':<14}{'누적 전체'}")
print("-" * 70)

context = 0
total_in = 0
total_out = 0

for turn in range(1, 9):
    context += user_tokens                 # 이번 질문 추가
    this_input = context                   # 지금까지 전체를 보낸다
    total_in += this_input
    total_out += assistant_tokens
    context += assistant_tokens            # 모델 답변도 기록에 추가

    if turn in (1, 2, 3, 5, 8):
        print(f"{turn:<6}{this_input:<14}{assistant_tokens:<14}"
              f"{total_in:<14}{total_in+total_out}")

print("-" * 70)
print()
print(f"8턴 대화에서 입력 토큰만 {total_in:,}개가 쌓였다.")
print(f"매 턴 50토큰씩만 썼는데도 {total_in/(user_tokens*8):.1f}배가 되었다.")
print()
print("이것이 대화형 API의 비용 구조다.")
print()
print("대응 방법")
print("  1) 오래된 대화는 잘라낸다 (최근 N턴만 유지)")
print("  2) 이전 대화를 요약해 짧게 대체한다")
print("  3) system 메시지를 간결하게 유지한다")

In [ ]:
print("=" * 70)
print("비용 계산 함수")
print("=" * 70)


def estimate_cost(prompt_tokens, completion_tokens,
                  input_price_per_1m, output_price_per_1m):
    """토큰 수와 단가로 비용을 계산한다.

    단가는 보통 '100만 토큰당 달러'로 표시된다.
    실제 단가는 제공처 문서에서 확인해야 하며 자주 바뀐다.
    """
    cost_in = prompt_tokens / 1_000_000 * input_price_per_1m
    cost_out = completion_tokens / 1_000_000 * output_price_per_1m
    return cost_in + cost_out, cost_in, cost_out


# 예시 단가 (실제 값이 아님 — 계산 방법을 보이기 위한 가정)
EXAMPLE_IN = 0.15    # 100만 입력 토큰당 $0.15 라고 가정
EXAMPLE_OUT = 0.60   # 100만 출력 토큰당 $0.60 라고 가정

print(f"가정한 단가: 입력 ${EXAMPLE_IN}/1M, 출력 ${EXAMPLE_OUT}/1M")
print("(실제 단가는 제공처 문서에서 확인하세요 — 자주 바뀝니다)")
print()

scenarios = [
    ("짧은 질문 1회",      50, 200),
    ("8턴 대화",           total_in, total_out),
    ("하루 1000회 호출",   50*1000, 200*1000),
]

print(f"{'상황':<22}{'입력':<12}{'출력':<12}{'비용(가정)'}")
print("-" * 70)
for name, pt, ct in scenarios:
    total, ci, co = estimate_cost(pt, ct, EXAMPLE_IN, EXAMPLE_OUT)
    print(f"{name:<22}{pt:<12,}{ct:<12,}${total:.4f}")
print("-" * 70)
print()
print("계산 자체는 단순하다. 중요한 것은 **토큰이 얼마나 빨리 쌓이는가**다.")

---

## 7. 구조화된 출력 — 이론편 20.6절

24장 7절에서 프롬프트로 JSON 형식을 요청하는 방법을 봤다.
API에는 **형식을 강제하는 기능**이 있는 경우가 있다.

In [ ]:
import json

print("=" * 65)
print("JSON 형식으로 답하게 하기")
print("=" * 65)

system_prompt = """당신은 텍스트에서 정보를 추출해 JSON으로만 답합니다.
형식: {"name": "이름", "age": 숫자, "job": "직업"}
다른 설명은 덧붙이지 마세요."""

user_text = "김철수 씨는 32세이고 소프트웨어 개발자로 일하고 있습니다."

msgs = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_text},
]

print(f"입력: {user_text}")
print()

# 방법 1: 프롬프트로 요청 (24장 7절)
resp = ask(msgs, temperature=0, max_tokens=150)
if resp:
    raw = resp.choices[0].message.content
    print("[프롬프트로 요청]")
    print(f"  원본 응답: {raw}")

    # 24장에서 만든 파싱 함수와 같은 방식
    cleaned = raw.strip().replace("```json", "").replace("```", "").strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]
    try:
        data = json.loads(cleaned)
        print(f"  파싱 성공: {data}")
        print(f"  나이만 꺼내기: {data.get('age')}")
    except json.JSONDecodeError as e:
        print(f"  파싱 실패: {e}")

In [ ]:
print("=" * 65)
print("response_format — 형식을 강제하는 기능")
print("=" * 65)
print()
print("일부 제공처는 JSON 출력을 보장하는 옵션을 제공한다.")
print()
print("""
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    response_format={"type": "json_object"},   # JSON 보장
)
""")
print()
print("이 옵션을 쓰면")
print("  - 반드시 유효한 JSON이 나온다")
print("  - 코드 블록이나 설명이 붙지 않는다")
print("  - 파싱 실패를 걱정하지 않아도 된다")
print()
print("주의사항")
print("  - 프롬프트에도 'JSON으로 답하라'고 명시해야 하는 경우가 많다")
print("  - 모든 제공처·모델이 지원하지는 않는다")
print("  - 지원 여부는 문서에서 확인한다")
print()

# 실제 시도 (지원하지 않으면 오류 처리)
resp = ask(msgs, temperature=0, max_tokens=150,
           response_format={"type": "json_object"})
if resp:
    import json
    raw = resp.choices[0].message.content
    print("-" * 65)
    print(f"결과: {raw}")
    try:
        print(f"파싱: {json.loads(raw)}")
        print("[OK] 파싱 없이 바로 쓸 수 있다")
    except Exception as e:
        print(f"파싱 실패: {e}")

---

## 8. 오류 처리와 재시도

API 호출은 **실패할 수 있다.** 네트워크 문제, 사용량 초과, 서버 과부하 등이다.

실무 코드에는 대응이 필요하다.

| 오류 | 원인 | 대응 |
|---|---|---|
| `RateLimitError` | 호출 한도 초과 | 잠시 기다렸다 재시도 |
| `AuthenticationError` | 키가 잘못됨 | 키 확인 (재시도 무의미) |
| `APIConnectionError` | 네트워크 문제 | 재시도 |
| `BadRequestError` | 요청 형식 오류 | 코드 수정 (재시도 무의미) |

**재시도해도 되는 것과 안 되는 것을 구분하는 것**이 핵심이다.

In [ ]:
import time
import random


def ask_with_retry(messages, max_retries=3, base_delay=1.0, **kwargs):
    """지수 백오프로 재시도하는 호출 함수

    지수 백오프: 실패할 때마다 대기 시간을 2배씩 늘린다.
      1초 → 2초 → 4초 ...
    서버가 과부하일 때 모두가 동시에 재시도하면 상황이 나빠지므로,
    무작위 지연(jitter)도 섞는다.
    """
    client = get_client()
    if client is None:
        print("[건너뜀] API 키가 없습니다.")
        return None

    for attempt in range(max_retries):
        try:
            return client.chat.completions.create(
                model=DEFAULT_MODEL, messages=messages, **kwargs)

        except Exception as e:
            name = type(e).__name__

            # 재시도가 무의미한 오류
            if name in ("AuthenticationError", "BadRequestError",
                        "PermissionDeniedError", "NotFoundError"):
                print(f"[중단] {name} — 재시도해도 해결되지 않습니다.")
                print(f"  {str(e)[:150]}")
                return None

            # 재시도 가능한 오류
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
                print(f"[재시도 {attempt+1}/{max_retries}] {name} — "
                      f"{delay:.1f}초 후 다시 시도")
                time.sleep(delay)
            else:
                print(f"[실패] {max_retries}회 시도 후 포기: {name}")
                print(f"  {str(e)[:150]}")
                return None


print("=" * 65)
print("재시도 로직 확인")
print("=" * 65)

resp = ask_with_retry(
    [{"role": "user", "content": "1+1은?"}],
    max_tokens=30)

if resp:
    print(f"응답: {resp.choices[0].message.content}")

print()
print("-" * 65)
print("지수 백오프 대기 시간")
print(f"{'시도':<10}{'대기 시간'}")
for i in range(4):
    print(f"{i+1:<10}{1.0 * (2**i):.1f}초")
print()
print("실무에서는 tenacity 같은 라이브러리를 쓰면 더 간단하다.")

In [ ]:
print("=" * 65)
print("키가 유출되었을 때")
print("=" * 65)
print()
print("만약 키를 실수로 저장소에 올렸다면")
print()
print("  1) 즉시 해당 키를 **삭제(revoke)** 한다")
print("     - 커밋을 되돌리는 것만으로는 부족하다")
print("     - 이력에 남아 있고, 이미 수집되었을 수 있다")
print()
print("  2) 새 키를 발급받는다")
print()
print("  3) 사용량을 확인해 이상 호출이 없는지 본다")
print()
print("  4) 사용량 한도를 설정해 둔다")
print()
print("-" * 65)
print("예방이 훨씬 쉽다")
print("  - .env 에만 두고 .gitignore 에 등록")
print("  - 커밋 전 git diff 로 확인하는 습관")
print("  - GitHub 의 비밀정보 검사 기능 활성화")
print()

# .gitignore 확인
from pathlib import Path
root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent

gitignore = root / ".gitignore"
if gitignore.exists():
    content = gitignore.read_text(encoding="utf-8")
    has_env = ".env" in content
    print(f"이 저장소의 .gitignore 에 .env 포함: {'예' if has_env else '아니오'}")
    if not has_env:
        print("  → .gitignore 에 .env 를 추가하세요")
else:
    print(".gitignore 파일이 없습니다. 만들어 .env 를 등록하세요.")

---

## 9. 정리

### 로컬 모델과 API — 언제 무엇을

| 상황 | 권장 |
|---|---|
| 민감한 데이터 | 로컬 (23~24장) |
| 최고 성능이 필요 | API |
| 대량 반복 처리 | 로컬 (비용) |
| 빠른 시작 | API |
| 파인튜닝 필요 | 로컬 (28~32장) |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 키 보관 | **`.env`에만**, 코드에 직접 쓰지 않기 |
| `base_url` | 이것만 바꾸면 제공처 교체 가능 |
| 대화 기록 | API는 기억하지 않음 — 매번 전체 전송 |
| 비용 | 대화가 길어지면 급격히 증가 |
| `finish_reason` | `length`면 답이 잘린 것 |
| 스트리밍 | 체감 대기 시간 크게 감소 |
| 재시도 | 재시도 가능한 오류만 구분해서 |

### 24장과 이어지는 지점

| 20번 (로컬) | 21번 (API) |
|---|---|
| ChatML 문자열 직접 구성 | `messages` 리스트 |
| `temperature`, `top_p` | 같은 이름, 같은 개념 |
| `max_new_tokens` | `max_tokens` |
| 토큰 = 문맥 창 소비 | 토큰 = **비용** |

**개념은 같고 표현만 다르다.** 로컬에서 원리를 익혔기 때문에
API 문서를 읽을 때도 무엇을 뜻하는지 바로 알 수 있다.

### 다음 장

**26. 토크나이저 직접 학습 — BPE 구현** — 이론편 20.2절.
텍스트를 벡터로 바꿔 의미가 비슷한 것을 찾는 방법을 다룬다.
**이론편 20.2절에서 손으로 계산한 BPE 병합 과정**을 확인한다.

### 비용 구조를 그림으로

6절에서 계산한 토큰 누적을 그래프로 보면 **왜 급격히 늘어나는지** 한눈에 보인다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

user_tokens, assistant_tokens = 50, 200

# --- 왼쪽: 턴에 따른 누적 ---
ax = axes[0]
turns = range(1, 13)
context, total_in, history_in, history_this = 0, 0, [], []

for _ in turns:
    context += user_tokens
    history_this.append(context)
    total_in += context
    history_in.append(total_in)
    context += assistant_tokens

ax.plot(list(turns), history_this, marker="o", linewidth=2,
        color="#EA580C", label="이번 턴 입력")
ax.plot(list(turns), history_in, marker="s", linewidth=2,
        color="#DC2626", label="누적 입력")
ax.set_xlabel("대화 턴")
ax.set_ylabel("입력 토큰")
ax.set_title("대화가 길어지면")
ax.legend()
ax.grid(alpha=0.3)

# --- 오른쪽: 대응 방법별 비교 ---
ax = axes[1]
strategies = {
    "전체 유지": history_in[-1],
    "최근 4턴만": sum(history_this[-4:]) + sum(history_this[:4]),
    "요약 사용": int(history_in[-1] * 0.45),
    "매번 새 대화": (user_tokens) * len(list(turns)),
}
colors = ["#DC2626", "#EA580C", "#0D9488", "#1E40AF"]
bars = ax.bar(range(len(strategies)), list(strategies.values()), color=colors)
for b, v in zip(bars, strategies.values()):
    ax.text(b.get_x() + b.get_width()/2, v + 300, f"{v:,}",
            ha="center", fontsize=8)
ax.set_xticks(range(len(strategies)))
ax.set_xticklabels(list(strategies.keys()), fontsize=8, rotation=12, ha="right")
ax.set_ylabel("12턴 후 누적 입력 토큰")
ax.set_title("대응 방법별 비교")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("왼쪽: 매 턴 50토큰씩만 써도 누적은 가파르게 오른다")
print(f"  12턴이면 {history_in[-1]:,} 토큰")
print()
print("오른쪽: '매번 새 대화'가 가장 싸지만 문맥을 잃는다")
print("  대부분은 '최근 N턴 유지'나 '요약' 사이에서 균형을 잡는다.")